In [1]:
from helpers.persistence import save_var, load_var
from helpers.progress_bar import ProgressBar
from helpers.openml_data_v2 import get_data1, openml_cc18_list, hard_list
from helpers.openml_data import tabular_id_list

In [2]:
import numpy as np
from tqdm import tqdm
import scipy

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import pairwise_distances, pairwise_distances_chunked

In [4]:
from NPT.run import main
from NPT.npt.configs import build_parser

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


In [5]:
# npt uses sklearn to do CV and splitting and uses the same random state = 42 and has same test_index
# problem is that somewhere in trainer or deeper the data rows are shuffled and that is the y_preds assertion fails

In [6]:
# dataset_name = 23

# parser = build_parser()
# args = parser.parse_args([
#     '--data_set', f'custom__{dataset_name}', 
#     '--custom_data_set', f'{dataset_name}', 
#     '--exp_test_perc', '0.2',
#     '--exp_val_perc', '0.1',
#     '--exp_patience', '30',
#     '--exp_n_runs', '1',
#     '--exp_num_total_steps', '10',
#     '--exp_batch_size', '128',
#     # '--exp_disable_cuda',
#     # '--data_set_on_cuda', 'True',
#     '--exp_full_batch_gd',
# ])


# fold_preds, fold_trues = main(args)
# # dataset, _ = main(args)
# # args

In [7]:
def get_cv_results(test_preds, test_trues):
    
    scores = []
    y_trues = []
    y_preds = []
    
    
    # pbar.add_prefix('starting 10-fold cv')

    for fold_index, (y_test, y_pred) in enumerate(zip(test_preds, test_trues)):    
        
        score = f1_score(y_test, y_pred, average='weighted')
        
        # acc = ((y_test == y_pred).sum() / y_test.shape[0])
        # print('acc', acc)
        
        scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(y_test)
        
    return scores, y_preds, y_trues


# scores, _, _ = get_cv_results(fold_preds, fold_trues)
# scores

In [8]:
save_path = './saved_vars/test-npt.pkl'
dataset_results = load_var(save_path) or {}

cache_path = './saved_vars/test-npt-outputs.pkl'
outputs = load_var(cache_path) or {}

# dataset_results, outputs = {}, {}

In [8]:
save_path = './saved_vars/test-npt-time.pkl'
dataset_results = load_var(save_path) or {}

cache_path = './saved_vars/test-npt-outputs-time.pkl'
outputs = load_var(cache_path) or {}

# dataset_results, outputs = {}, {}

In [9]:
# pbar = ProgressBar(data_loaders.items())
pbar = hard_list
pbar = [ 1063,  1510,  1464,   469,   458,  1494,  1068,  1049,    23,
        1050, 40975, 40982,  1067,  1487,  1485,  4134, 40701,  1497,
        1475,  4538]

pbar = [ 458, 1050, 1475, 1485, 1487, 1497, 4134, 4538][::2]

skip_list = [1050, 1487] #outOfMemory error

# pbar = [1485]
pbar = [1510]

pbar = [i for i in pbar if i not in [458]]
# pbar = [('breast cancer', data_loaders['breast cancer'])]
failed_list = []

for dataset_name in pbar:
    
    # if dataset_name in skip_list:
    #     continue
    
    try:
        df, y, cat = get_data1(dataset_name)
    except Exception as e:
        print('couldnt do', dataset_name)
        # raise e
        failed_list.append(dataset_name)
        continue
        
    if df.shape[1] > 100:
        continue
        
    batch_size = 128
    # NPT takes max steps as input and calculates epochs from there.
    n_batches = int(np.ceil(df.shape[0]*0.8/batch_size)) if batch_size > 0 else 1
    n_steps = n_batches * 1000
    # print(n_steps)

    parser = build_parser()
    args = parser.parse_args([
        '--data_set', f'custom__{dataset_name}', 
        '--custom_data_set', f'{dataset_name}', 
        '--exp_test_perc', '0.2',
        '--exp_val_perc', '0.1',
        '--exp_patience', '-1',
        '--exp_n_runs', '30',
        '--exp_bootstrap', '30',
        '--exp_num_total_steps', f'{n_steps}',
        '--exp_batch_size', f'{batch_size}',
        # '--exp_disable_cuda',
    ])
    
    print('Trying ', dataset_name)
    
    fold_preds, fold_trues = [], []
    
    if dataset_name in outputs.keys():
        fold_preds, fold_trues = outputs[dataset_name]
        print(f'NPT output for {dataset_name} found, skipping model train & eval')
    else:
        fold_preds, fold_trues = main(args)
        outputs[dataset_name] = fold_preds, fold_trues
        save_var(outputs, cache_path)
    
    try:
        if dataset_name in dataset_results.keys():
            print(f'NPT results for {dataset_name} found, skipping loop')
            continue
            
        scores = get_cv_results(fold_preds, fold_trues)
        dataset_results[dataset_name] = scores
        save_var(dataset_results, save_path)

    except Exception as e:
        print('couldnt do', dataset_name)
        # raise e
        failed_list.append(dataset_name)
    
    # print(scores)

2023-08-24 12:41:22 | INFO | openml.datasets.dataset | pickle load data ozone-level-8hr


Trying  1487
Configuring arguments...
Doing k-FOLD CV. Assigning group name 35dqrqc8.
Running model with CUDA
data_set_on_cuda False


CV Splits for this dataset are cached. Loading from file.
CV Index: 0
Train-test Split 1/30
Building NPT.
Using feature type embedding (unique embedding for categorical and numerical features).
Using feature index embedding (unique embedding for each column).
Clipping gradients to value 1.0.
Model has 1135536266 parameters,batch size 128.
Initialized "lookahead_lamb" optimizer.
Warming up for 11200.0/16000.0 steps.
Initialized "flat_and_anneal" learning rate scheduler.
Initialized "cosine" augmentation/label tradeoff annealer. Annealing to minimum value in 16000 steps.
Using AUROC in loss module.


/home/cida-lab-2/miniconda3/envs/npt/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:138: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


OutOfMemoryError: CUDA out of memory. Tried to allocate 334.00 MiB (GPU 0; 15.74 GiB total capacity; 14.54 GiB already allocated; 181.88 MiB free; 14.55 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
failed_list

In [ ]:
outputs.keys(), dataset_name

In [ ]:
# outputs[dataset_name] = fold_preds, fold_trues
# save_var(outputs, cache_path)

In [ ]:
np.mean(dataset_results[23][0])